In [1]:
from sarma.ingestion.loader import load_pdf
from sarma.ingestion.splitter import split_documents
from sarma.vectorstore.vectorstore import create_vector_store
from sarma.retriever import create_retriever
from sarma.rag.rag import create_rag_chain
from sarma.llm import llm

C:\Users\rost8\anaconda3\envs\sarma\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 199/199 [00:00<00:00, 6295.85it/s]


In [2]:
documents = load_pdf("../data/raw/RB209 Arable crops.pdf")
chunks = split_documents(documents)
db = create_vector_store(chunks)
retriever = create_retriever(db)
chain = create_rag_chain(retriever)

In [9]:
response = chain.invoke("What is the best Nitrogen value for wheat?")
print(response.content)

The best nitrogen value for wheat depends on the purpose:  
- **Feed wheat**: The economic optimum nitrogen concentration is **1.9% N** (corresponding to 11% protein).  
- **Bread-making wheat**: The economic optimum nitrogen concentration is **2.1% N** (corresponding to 12% protein).  

These values are derived from grain protein concentrations, which are adjusted based on nitrogen application rates and economic factors.


In [10]:
results = db.similarity_search_with_score(
"What is the best Nitrogen value for wheat?",
k=10
)

for doc, score in results:
    print('SCORE:', score)
    print(doc.page_content[:200])
    print("="*50)

SCORE: 0.3302615284919739
Return to Contents
Wheat – use of grain nitrogen concentration
Farm nitrogen strategies for wheat can be assessed periodically using information 
on grain protein concentration. Grain protein at the e
SCORE: 0.3302615284919739
Return to Contents
Wheat – use of grain nitrogen concentration
Farm nitrogen strategies for wheat can be assessed periodically using information 
on grain protein concentration. Grain protein at the e
SCORE: 0.3806183934211731
The recommendations in the tables for wheat and barley are based on a 
breakeven ratio of 5.0 (cost of fertiliser nitrogen as £/kg N divided by value  
of grain as £/kg). If the price of nitrogen or t
SCORE: 0.3806183934211731
The recommendations in the tables for wheat and barley are based on a 
breakeven ratio of 5.0 (cost of fertiliser nitrogen as £/kg N divided by value  
of grain as £/kg). If the price of nitrogen or t
SCORE: 0.385998010635376
32
Cereals
Wheat, spring sown – nitrogen
Table 4.18 Nitrogen for sp

In [11]:
print(db._collection.count())

330


In [12]:
items = db._collection.get(
    limit=10,
    include=["metadatas", "documents"]
)

for meta in items["metadatas"]:
    print(meta)

{'page': 1, 'source': 'RB209 Arable crops.pdf', 'moddate': '2017-12-06T09:57:17+00:00', 'trapped': '/False', 'creator': 'Adobe InDesign CC 13.0 (Macintosh)', 'producer': 'Adobe PDF Library 15.0', 'creationdate': '2017-12-06T09:57:02+00:00', 'page_label': '4S1', 'total_pages': 52}
{'producer': 'Adobe PDF Library 15.0', 'moddate': '2017-12-06T09:57:17+00:00', 'total_pages': 52, 'creationdate': '2017-12-06T09:57:02+00:00', 'trapped': '/False', 'page_label': '4S3', 'page': 3, 'creator': 'Adobe InDesign CC 13.0 (Macintosh)', 'source': 'RB209 Arable crops.pdf'}
{'moddate': '2017-12-06T09:57:17+00:00', 'page': 3, 'source': 'RB209 Arable crops.pdf', 'creationdate': '2017-12-06T09:57:02+00:00', 'total_pages': 52, 'trapped': '/False', 'producer': 'Adobe PDF Library 15.0', 'creator': 'Adobe InDesign CC 13.0 (Macintosh)', 'page_label': '4S3'}
{'page_label': '4S3', 'creator': 'Adobe InDesign CC 13.0 (Macintosh)', 'page': 3, 'total_pages': 52, 'source': 'RB209 Arable crops.pdf', 'creationdate': '201

In [13]:
response1 = chain.invoke(
    "What nitrogen rate is recommended for spring sown wheat?"
)

print(response1.content)

The recommended nitrogen rates for spring sown wheat depend on soil type, as outlined in **Table 4.18**:

- **Light sand soils**: 160, 130, 100, 70, 40, 0–40, 0 kg N/ha  
- **All other mineral soils**: 210a, 180, 150, 120, 70, 40, 0–40 kg N/ha  
- **Organic soils**: 120, 70, 40, 0–40 kg N/ha  
- **Peaty soils**: 0–40 kg N/ha  

Note: The "a" in 210a indicates the recommendation exceeds the nitrogen maximum limit in Nitrate Vulnerable Zones (NVZs), which applies to the entire farm area for a crop type, not individual fields. For detailed guidance, refer to [www.gov.uk/nitrate-vulnerable-zones](https://www.gov.uk/nitrate-vulnerable-zones).
